# Logits Alignments

In [1]:
def default_params(): 
    return {
        'current_model': 'M1',
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/data/extension/mitigation/datasets',
            'current': 'base', # 'base' or 'prompted',
            'content_column': 'code',
            'sampling_size': 500,
            'prompt_column': 'prompt',
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'raw_logits_path' : '/workspaces/CodeSmells/datax/code_smells/logits/mitigation',
        'alignments_path': '/workspaces/CodeSmells/data/extension/mitigation/alignments',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            ##### BY ARCHITECTURE, SAME SIZE #####
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
            'M4' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b,
            ##### BY SIZE, SAME ARCHITECTURE #####
            'S1' : 'Qwen/Qwen2.5-Coder-0.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B,
            'S2' : 'Qwen/Qwen2.5-Coder-1.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B,
            'S3' : 'Qwen/Qwen2.5-Coder-3B', #https://huggingface.co/Qwen/Qwen2.5-Coder-3B,
            'S4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import random
import numpy as np
from statistics import mean, median
import os
import torch
import gc
from difflib import SequenceMatcher
from scipy.stats import entropy

In [3]:
from datasets import load_dataset, Dataset

In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

2025-07-02 15:41:45.761400: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751470905.779120  992705 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751470905.784644  992705 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-02 15:41:45.802594: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}"
create_folder(log_file)
log_file += '/align_aggr.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [7]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Model Loading

In [8]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir, use_fast=True)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     match params['quantization']:
               case 'int4':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
               case 'int8':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
               case 'float32':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
               case 'float16':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
               case _: 
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [9]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

#### Load Dataset

In [10]:
df_actual_ntp = pd.read_json(f"{params['raw_logits_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}/raw_logits.json")

In [11]:
df_actual_ntp.head(2)

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,s_end_line,s_end_column,s_code,category,input_lenght,input_ids,max_prob,min_prob,actual_prob,loss
0,2758,0e9baa6d2c7a752eb32e6da2359470f95a3efeec,PySyft,packages/syft/src/syft/oblv/model.py,model.py,get_uploaded_datasets,Adding method to get datasets list,def get_uploaded_datasets(self):\n if l...,https://github.com/OpenMined/PySyft.git,Python,...,9,100,"raise Exception(""User cannot connect to this d...",Warning,360,"[822, 679, 29918, 9009, 287, 29918, 14538, 169...","[[<PRE>, 0.7492900491000001], [module, 0.47943...","[[<s>, 0.0], [$}, 1e-10], [oreferrer, 0.0], [o...","[[def, 0.0007611316000000001], [get, 0.0267730...",1.795405
1,115917,6eb408a9973fbc24c973d6524dc34cb9b1e0ee05,mindsdb,mindsdb/api/mongo/responders/delete.py,delete.py,_result,del model interface,"def _result(self, query, request_env, mindsdb_...",https://github.com/mindsdb/mindsdb.git,Python,...,10,129,"raise Exception(""For db.predictors.delete oper...",Warning,292,"[822, 903, 2914, 29898, 1311, 29892, 2346, 298...","[[<PRE>, 0.7492886186000001], [module, 0.47944...","[[<s>, 0.0], [$}, 1e-10], [oreferrer, 0.0], [o...","[[def, 0.0007611339000000001], [_, 0.009010342...",1.161735


In [12]:
df_actual_ntp.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'fun_name',
       'commit_message', 'code', 'url', 'language', 'ast_errors',
       'n_ast_errors', 'ast_levels', 'n_whitespaces', 'n_words', 'vocab_size',
       'complexity', 'nloc', 'token_counts', 'n_ast_nodes', 'n_identifiers',
       's_msg_id', 's_line', 's_column', 's_end_line', 's_end_column',
       's_code', 'category', 'input_lenght', 'input_ids', 'max_prob',
       'min_prob', 'actual_prob', 'loss'],
      dtype='object')

#### Dataset Prompt Cleaning

In [13]:
def clean_row_from_prompt(row):
    row['code'] = row['original_code']
    row['input_ids'] = row['input_ids'][len(row['prompt_ids']):]
    row['max_prob'] = row['max_prob'][len(row['prompt_ids']):]
    row['min_prob'] = row['min_prob'][len(row['prompt_ids']):]
    row['actual_prob'] = row['actual_prob'][len(row['prompt_ids']):]
    return row

In [14]:
if params['dataset']['current'] == 'prompted':
    # Remove the prompt from the code and logits
    df_actual_ntp = df_actual_ntp.apply(lambda row: clean_row_from_prompt(row), axis=1)

In [15]:
df_actual_ntp = df_actual_ntp.reset_index(drop=True)

In [16]:
assert len(df_actual_ntp.loc[0]['input_ids']) == len(df_actual_ntp.loc[0]['max_prob']) == len(df_actual_ntp.loc[0]['min_prob']) == len(df_actual_ntp.loc[0]['actual_prob'])

#### Token Binding

In [17]:
def find_range_of_indexes(positions, search_range):
    """
    Finds the range of indexes in the positions array where the search_range is fully included.
    
    Args:
    - positions: A list of tuples, where each tuple is (start_position, end_position) (inclusive).
    - search_range: A tuple (start_position, end_position), where start_position is inclusive and end_position is exclusive.
    
    Returns:
    - A tuple (start_index, end_index) representing the range of indexes in the positions array where the search_range is included.
    """
    start, end = search_range
    start_index = -1
    end_index = -1

    for i, (pos_start, pos_end) in enumerate(positions):
        if pos_start <= start <= pos_end:  # Find the start of the range
            start_index = i
        if pos_start <= end - 1 <= pos_end and pos_end>=end:  # Find the end of the range
            end_index = i
            break

    if start_index != -1 and end_index != -1:
        return (start_index, end_index)
    else:
        return None  # If no range is found

In [18]:
def get_substring_positions(code: str, code_smell: str, start):
    """
    Calculate the start and end positions of the substring based on line and column information.

    Parameters:
    text (str): The input string containing multiple lines.
    start (tuple): A tuple of (start_line, start_column) indicating the start position.
    end (tuple): A tuple of (end_line, end_column) indicating the end position.

    Returns:
    tuple: A tuple containing (start_position, end_position) of the substring in the input string.
    """
    lines = code.split('\n')  # Split the string into lines

    # Calculate the character position for the start of the substring
    start_line, start_column = start
    
    try:
        start_position = sum(len(lines[i]) + 1 for i in range(start_line - 1)) + start_column
    except:
        start_position = code.find(code_smell)

    if start_line > len(lines) or start_position>= len(code): 
        start_position = code.find(code_smell)

    if start_position == -1:
        match = SequenceMatcher(None, code, code_smell).find_longest_match()
        start_position= match.a
        end_position = match.a + match.size
    else:
        end_position = start_position + len(code_smell)
        end_position = len(code) if end_position >= len(code) else end_position
    

    return (start_position, end_position)

In [19]:
def find_code_smell_logits(code, code_smell_pos, logits_array, tokenizer):
    indexes_range = find_range_of_indexes(tokenizer.encode_plus(code, return_offsets_mapping=True, add_special_tokens=False)['offset_mapping'], code_smell_pos)
    return logits_array[indexes_range[0]:indexes_range[1]+1]

In [20]:
df_actual_ntp.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'fun_name',
       'commit_message', 'code', 'url', 'language', 'ast_errors',
       'n_ast_errors', 'ast_levels', 'n_whitespaces', 'n_words', 'vocab_size',
       'complexity', 'nloc', 'token_counts', 'n_ast_nodes', 'n_identifiers',
       's_msg_id', 's_line', 's_column', 's_end_line', 's_end_column',
       's_code', 'category', 'input_lenght', 'input_ids', 'max_prob',
       'min_prob', 'actual_prob', 'loss'],
      dtype='object')

In [21]:
df_actual_ntp['code_smell_pos'] = df_actual_ntp.apply(lambda row: get_substring_positions(row['code'], row['s_code'], (row['s_line'], row['s_column'])), axis=1)

In [22]:
df_actual_ntp.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'fun_name',
       'commit_message', 'code', 'url', 'language', 'ast_errors',
       'n_ast_errors', 'ast_levels', 'n_whitespaces', 'n_words', 'vocab_size',
       'complexity', 'nloc', 'token_counts', 'n_ast_nodes', 'n_identifiers',
       's_msg_id', 's_line', 's_column', 's_end_line', 's_end_column',
       's_code', 'category', 'input_lenght', 'input_ids', 'max_prob',
       'min_prob', 'actual_prob', 'loss', 'code_smell_pos'],
      dtype='object')

#### Aggregation Functions

In [23]:
def compute_relative_psc(actual_probs, min_probs, max_probs, epsilon=1e-9):
    # Normalize probabilities
    relative_probs = (actual_probs - min_probs) / (max_probs - min_probs + epsilon)

    # Compute relative PSC as the mean of relative probabilities
    relative_psc = np.mean(relative_probs)
    
    return relative_psc

In [24]:
def compute_psc_entropy_scaled(actual_probs, min_probs, max_probs, temperature=1.0, epsilon=1e-9):
    # Apply temperature scaling
    scaled_probs = np.exp(actual_probs / temperature) / np.sum(np.exp(actual_probs / temperature))

    # Compute Shannon entropy per token
    entropy_scores = entropy(scaled_probs, base=2)  # Base 2 for information entropy

    # Normalize using entropy-based weighting
    entropy_norm = 1 - (entropy_scores / np.log2(len(actual_probs) + epsilon))  # Normalize between 0 and 1

    # Compute final PSC score (weighted by entropy)
    adjusted_psc = np.mean(entropy_norm * scaled_probs)

    return adjusted_psc

#### Execute

In [25]:
# Alignments
df_actual_ntp['code_smell_actual_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['actual_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_max_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['max_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_min_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['min_prob'], tokenizer), axis=1)

In [26]:
df_actual_ntp = df_actual_ntp[df_actual_ntp['code_smell_actual_logits'].apply(lambda x: len(x) > 0)].copy()
df_actual_ntp = df_actual_ntp[df_actual_ntp['code_smell_max_logits'].apply(lambda x: len(x) > 0)].copy()
df_actual_ntp = df_actual_ntp[df_actual_ntp['code_smell_min_logits'].apply(lambda x: len(x) > 0)].copy()
df_actual_ntp = df_actual_ntp.dropna().copy()

In [27]:
## Aggregations - median
df_actual_ntp['code_smell_actual_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [28]:
## Aggregations  - mean
df_actual_ntp['code_smell_actual_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [29]:
## Aggregations - entropy scaled
df_actual_ntp['code_smell_psc_entropy'] = df_actual_ntp.apply(lambda row: compute_psc_entropy_scaled(np.array([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']])), axis=1)
## Aggregations - normalized relative probabilities
df_actual_ntp['code_smell_psc_relative'] = df_actual_ntp.apply(lambda row: compute_relative_psc(np.array([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']])), axis=1)

In [30]:
df_actual_ntp[['code', 's_code', 's_msg_id', 'code_smell_psc_relative']]

,code,s_code,s_msg_id,code_smell_psc_relative
0,def get_uploaded_datasets(self):\n if l...,"raise Exception(""User cannot connect to this d...",W0719,0.712570
1,"def _result(self, query, request_env, mindsdb_...","raise Exception(""For db.predictors.delete oper...",W0719,0.825696
2,def get_uploaded_datasets(self):\n if l...,"raise Exception(""Either proxy not running or n...",W0719,0.633605
3,"def get_handler(self, name, company_id=None, c...","raise Exception(f""Cant find handler for '{inte...",W0719,0.782247
4,"def select(self, query):\n result = sel...",raise Exception(result.error_message),W0719,0.728480
...,...,...,...,...
495,"def add(self, name, query, integration_name, c...","raise Exception(f""Can't find integration with ...",W0719,0.894697
496,def usbonds_command():\n \n\n # Debug us...,"raise Exception(""No available data found"")",W0719,0.915459
497,"def request_publish(self, dataset_id, sigma = ...","raise Exception(""No Domain Clients added. Set ...",W0719,0.572874
498,"def _learn(self, statement):\n model_na...","raise Exception(""Ludwig handler does not suppo...",W0719,0.672969


#### SAVE

In [31]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [32]:
alignments_dir = f"{params['alignments_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
create_folder(alignments_dir)
df_actual_ntp.to_json(f"{alignments_dir}/aligned_smells.json")

In [33]:
torch.cuda.empty_cache()
gc.collect()

0